# Agent RAG — BCMID Breast Cancer Multimodal AI
## Architecture : Data Quality → Vision → RAG → Raisonnement → Contrôle qualité → Rapport

**Notebook v3 — version corrigée et complète (Kaggle-exécutable)**

### Corrections apportées
1. BI-RADS et confiance : aucune valeur fabriquée ; valeurs manquantes → `null` + motif
2. Isolation des modalités : étanchéité mathématiquement vérifiée
3. RAG : exclusion patient / groupe / pli avant classement
4. Raisonnement : probabilités vision inchangées ; mode extractif explicitement signalé
5. Contrôle qualité : statuts explicites ; revue humaine distincte
6. XAI : heatmaps vérifiées ; "indisponible" si absentes
7. Kaggle : diagnostic préalable, chemins centralisés, CPU/GPU auto-détecté

### Avertissement clinique
Ce notebook est un prototype de recherche. Il n'est pas validé cliniquement.
Toutes les sorties requièrent la revue d'un radiologue.

---


## R0 : Diagnostic des fichiers, dépendances et configuration

In [3]:
# ============================================================
# R0 : Diagnostic des fichiers, dépendances et configuration
# ============================================================
import subprocess, sys, json, re, os
from pathlib import Path
import numpy as np, pandas as pd

# ── 1. Installation des dépendances manquantes ──
for mod, pkg in [("sentence_transformers", "sentence-transformers"), ("faiss", "faiss-cpu")]:
    try:
        __import__(mod); print(f"{pkg:<24} déjà installé")
    except ImportError:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                          capture_output=True, text=True)
        print(f"{pkg:<24} " + ("installé" if r.returncode == 0 else "ÉCHEC : " + r.stderr[-200:]))

# ── 2. Détection environnement ──
IS_KAGGLE = Path("/kaggle").exists()
if IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    SEARCH_BASES = [Path("/kaggle/input"), Path("/kaggle/working")]
else:
    # Local Windows fallback
    _loc = Path(r"c:\Users\moham\OneDrive - ESPRIT\Documents\deeplearning5eme")
    WORK_DIR = _loc / "bcmid_work"
    SEARCH_BASES = [WORK_DIR / "vision_outputs", WORK_DIR / "dq_outputs",
                    WORK_DIR, _loc,
                    Path(r"c:\Users\moham\Downloads")]
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nEnvironnement : {'Kaggle' if IS_KAGGLE else 'Local'} | Répertoire de travail : {WORK_DIR}")

# ── 3. Recherche robuste des fichiers ──
def find_file(pattern, is_dir=False):
    for base in SEARCH_BASES:
        if not base.exists():
            continue
        for p in base.rglob(pattern):
            if is_dir and p.is_dir():    return p
            elif not is_dir and p.is_file(): return p
    return None

# ── 4. Localisation des fichiers requis ──
REP_DIR   = find_file("reports_anon", is_dir=True)
DS_PATH   = find_file("dataset_final.csv")
PRED_PATH = find_file("predictions_patient.csv")
FICHE_PATH= find_file("heatmap_fiches_pli0.json")
BLIND_CSV = find_file("reports_blind_clean.csv")   # peut déjà exister

print("\n── Fichiers détectés ──")
for nom, p in [("reports_anon/",     REP_DIR),
               ("dataset_final.csv", DS_PATH),
               ("predictions.csv",   PRED_PATH),
               ("heatmap_fiches",    FICHE_PATH),
               ("reports_blind_clean", BLIND_CSV)]:
    status = "✓" if p else "✗ MANQUANT"
    print(f"  {nom:<26} {status} {p or ''}")

# ── 5. Arrêt propre si fichiers critiques manquants ──
MISSING = [n for n, p in [
    ("reports_anon/ OU reports_blind_clean.csv", REP_DIR or BLIND_CSV),
    ("dataset_final.csv",                        DS_PATH),
    ("predictions_patient.csv",                  PRED_PATH),
] if not p]
if MISSING:
    print("\n[ERREUR CRITIQUE] Fichiers manquants :")
    for m in MISSING: print(f"  - {m}")
    print("""
Sur Kaggle, ajoutez ces outputs comme inputs du notebook :
  • data-quality-clean  →  reports_anon/, dataset_final.csv
  • agent-vision22      →  predictions_patient.csv, heatmap_fiches_pli0.json
""")
    raise FileNotFoundError("Fichiers d'entrée manquants — voir ci-dessus")

print("\n✓ Tous les fichiers critiques sont présents.")

sentence-transformers    déjà installé
faiss-cpu                déjà installé

Environnement : Kaggle | Répertoire de travail : /kaggle/working

── Fichiers détectés ──
  reports_anon/              ✓ /kaggle/input/notebooks/ikbelhamdi/data-quality-clean/reports_anon
  dataset_final.csv          ✓ /kaggle/input/notebooks/ikbelhamdi/data-quality-clean/dataset_final.csv
  predictions.csv            ✓ /kaggle/input/notebooks/ikbelhamdi/agent-vision22/vision_outputs/predictions_patient.csv
  heatmap_fiches             ✓ /kaggle/input/notebooks/ikbelhamdi/agent-vision22/vision_outputs/heatmap_fiches_pli0.json
  reports_blind_clean        ✗ MANQUANT 

✓ Tous les fichiers critiques sont présents.


## R0b : Chargement des données

In [4]:
# ============================================================
# R0b : Chargement des données (rapports, prédictions, heatmaps)
# ============================================================

# ── Rapports patients ──
if BLIND_CSV and BLIND_CSV.stat().st_size > 1000:
    # Utiliser le CSV nettoyé s'il existe et est non-vide
    rep = pd.read_csv(str(BLIND_CSV))
    # Colonnes attendues : pseudo_id, group_pseudo, fold, text_blind [ou text_blind5], text_find, text_neutre
    if "text_neutre" in rep.columns:
        rep["text"] = rep["text_neutre"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    elif "text_blind" in rep.columns:
        rep["text"] = rep["text_blind"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    print(f"Rapports chargés depuis reports_blind_clean.csv : {len(rep)}")
elif REP_DIR:
    # Charger depuis les JSON bruts
    _rows = []
    for f in sorted(REP_DIR.glob("*.json")):
        try:
            obj = json.load(open(f, encoding="utf-8"))
            _rows.append(obj)
        except Exception as e:
            print(f"  [AVERTISSEMENT] {f.name} : {e}")
    _raw = pd.DataFrame(_rows)
    # Utiliser text_blind si disponible, sinon text_clean
    _tcol = "text_blind" if "text_blind" in _raw.columns else "text_clean"
    rep = _raw[["pseudo_id", _tcol]].rename(columns={_tcol: "text_blind"})
    # Fusionner avec les métadonnées
    meta = pd.read_csv(str(DS_PATH))[["pseudo_id", "group_pseudo", "fold"]]
    rep = rep.merge(meta, on="pseudo_id", how="left")
    rep["text"] = rep["text_blind"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    print(f"Rapports chargés depuis reports_anon/ : {len(rep)}")
else:
    raise FileNotFoundError("Impossible de charger les rapports patients")

# Vérifications de sécurité obligatoires
assert "pseudo_id"    in rep.columns, "Colonne pseudo_id manquante"
assert "group_pseudo" in rep.columns, "Colonne group_pseudo manquante"
assert "fold"         in rep.columns, "Colonne fold manquante"
assert "text"         in rep.columns, "Colonne text manquante"

rep["fold"] = rep["fold"].fillna(-1).astype(int)   # -1 = non étiqueté (BI-RADS 0)
rep["n_mots"] = rep["text"].str.split().str.len()

# Exclure les rapports trop courts
_before = len(rep)
rep = rep[rep.n_mots >= 40].drop_duplicates("pseudo_id").reset_index(drop=True)
print(f"Rapports après filtre (≥40 mots, 1 par patient) : {len(rep)} / {_before}")
print(f"Plis : {rep.fold.value_counts().sort_index().to_dict()}")
print(f"Longueur (mots) : min {rep.n_mots.min()}, médiane {int(rep.n_mots.median())}, max {rep.n_mots.max()}")

# ── Vérifications BI-RADS (zéro fuite dans le texte) ──
birads_re = r"bi-?\s?rads?\s*\(?\s*[0-6]"
n_birads = int(rep["text"].str.contains(birads_re, case=False, regex=True).sum())
print(f"\nChiffre BI-RADS dans text (doit être 0) : {n_birads}")
assert n_birads == 0, f"Fuite BI-RADS dans text : {n_birads} rapports affectés"

# ── Prédictions Vision ──
pred = pd.read_csv(str(PRED_PATH), index_col=0)
# Colonnes de labels INTERDITES dans pred
label_cols = [c for c in pred.columns if "label" in c.lower() or c.lower() in ("y", "pathology")]
assert len(label_cols) == 0, f"Colonnes de label détectées dans pred : {label_cols}"
print(f"\nPrédictions : {pred.shape[0]} patients, {pred.shape[1]} colonnes")
print(f"Colonnes : {list(pred.columns)}")

# ── Heatmaps ──
fiches = {}
if FICHE_PATH:
    fiches = json.load(open(str(FICHE_PATH), encoding="utf-8"))
    print(f"\nHeatmaps : {len(fiches)} patients")
else:
    print("\n[INFO] Pas de fichier heatmap — visual_explanations = 'indisponible'")

# ── Dataset principal (pour group_pseudo/fold si absent) ──
meta = pd.read_csv(str(DS_PATH))[["pseudo_id", "group_pseudo", "fold"]]
print(f"\nDataset principal : {len(meta)} patients")

Rapports chargés depuis reports_anon/ : 321
Rapports après filtre (≥40 mots, 1 par patient) : 320 / 321
Plis : {-1: 5, 0: 62, 1: 61, 2: 64, 3: 65, 4: 63}
Longueur (mots) : min 50, médiane 212, max 424

Chiffre BI-RADS dans text (doit être 0) : 0

Prédictions : 327 patients, 20 colonnes
Colonnes : ['p_normal_mg', 'p_benign_mg', 'p_malignant_mg', 'p_malignant_max_mg', 'n_img_mg', 'ecart_img_mg', 'entropie_mg', 'p_normal_us', 'p_benign_us', 'p_malignant_us', 'p_malignant_max_us', 'n_img_us', 'ecart_img_us', 'entropie_us', 'p_malignant_fusion', 'appareil_standard', 'predicted_birads', 'confidence', 'birads_description', 'clinical_action']

Heatmaps : 64 patients

Dataset principal : 332 patients


## R1 : Découpage, embeddings BGE-small, index FAISS

In [5]:
# ============================================================
# R1 : Découpage en morceaux, embeddings BGE-small, FAISS
# ============================================================
import time, faiss
from sentence_transformers import SentenceTransformer

def make_chunks(text, size=100, step=70):
    """Découpe un texte en morceaux chevauchants."""
    w, out, i = text.split(), [], 0
    while True:
        out.append(" ".join(w[i:i + size]))
        if i + size >= len(w): break
        i += step
    return out

# Construire les morceaux avec group_pseudo pour l'exclusion stricte
chunk_rows = []
for r in rep.itertuples():
    for j, c in enumerate(make_chunks(r.text)):
        chunk_rows.append(dict(
            pseudo_id   = r.pseudo_id,
            group_pseudo= r.group_pseudo,
            fold        = r.fold,
            chunk_idx   = j,
            text        = c
        ))
C = pd.DataFrame(chunk_rows)
print(f"Morceaux : {len(C)} | patients : {C.pseudo_id.nunique()}")
print(f"Distribution morceaux/patient : {C.groupby('pseudo_id').size().value_counts().sort_index().to_dict()}")

# Embeddings
t0 = time.time()
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
E = emb_model.encode(
    C.text.tolist(), batch_size=32,
    normalize_embeddings=True, show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(E.shape[1])
index.add(E)
print(f"\nEmbeddings : {E.shape} en {time.time()-t0:.0f} s")
print(f"Index FAISS : {index.ntotal} vecteurs")

# ── Sauvegarde ──
faiss.write_index(index, str(WORK_DIR / "rag_index.faiss"))
C.to_csv(str(WORK_DIR / "rag_chunks.csv"), index=False)
print("Sauvegardé : rag_index.faiss, rag_chunks.csv")

Morceaux : 995 | patients : 320
Distribution morceaux/patient : {1: 12, 2: 25, 3: 212, 4: 60, 5: 9, 6: 2}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Embeddings : (995, 384) en 64 s
Index FAISS : 995 vecteurs
Sauvegardé : rag_index.faiss, rag_chunks.csv


## R2 : Guidelines ACR BI-RADS + fonction `evidence()` (exclusion triple)

> [!NOTE]
> Les 9 entrées ACR ci-dessous sont des **résumés éditoriaux** du lexique BI-RADS 5e édition.
> Ils ne constituent PAS des documents officiels authentifiés.
> Pour un usage clinique réel, utiliser les PDFs officiels ACR.


In [6]:
# ============================================================
# R2 : Base de connaissances ACR BI-RADS + fonction evidence()
# ============================================================

# ── Fonction de recherche avec exclusion patient/groupe/pli ──
def search_cases(qvec, exclude_pid, exclude_group, exclude_fold, k=3, pool=500):
    """
    Recherche les cas similaires en EXCLUANT :
      - le même patient (pseudo_id)
      - le même groupe (group_pseudo) — évite la contamination intra-groupe
      - le même pli (fold) — évite la contamination croisée d'évaluation
    """
    sc, ix = index.search(qvec[None].astype("float32"), pool)
    out, seen = [], set()
    for s, i in zip(sc[0], ix[0]):
        row = C.iloc[i]
        # Triple exclusion obligatoire
        if row.pseudo_id == exclude_pid:                                   continue
        if row.group_pseudo == exclude_group and exclude_group is not None: continue
        if row.fold == exclude_fold and exclude_fold >= 0:                 continue
        if row.pseudo_id in seen:                                          continue
        seen.add(row.pseudo_id)
        out.append(dict(
            source        = f"{row.pseudo_id}#chunk{row.chunk_idx}",
            source_patient= row.pseudo_id,
            chunk_idx     = int(row.chunk_idx),
            score         = round(float(s), 3),
            passage       = row.text[:200]
        ))
        if len(out) == k: break
    return out

# ── Test de sécurité de l'exclusion ──
print("=== Test de sécurité : exclusion patient / groupe / pli ===")
rng = np.random.default_rng(42)
test_pats = rep[rep.fold >= 0].sample(min(30, len(rep[rep.fold>=0])), random_state=42)
viol_pid = viol_grp = viol_fold = 0
for r in test_pats.itertuples():
    # Trouver le premier chunk du patient
    qi = C.index[C.pseudo_id == r.pseudo_id][0]
    qv = E[qi]
    for res in search_cases(qv, r.pseudo_id, r.group_pseudo, r.fold, k=5):
        rc = C[C.pseudo_id == res["source_patient"]].iloc[0]
        viol_pid  += int(rc.pseudo_id   == r.pseudo_id)
        viol_grp  += int(rc.group_pseudo == r.group_pseudo and r.group_pseudo is not None)
        viol_fold += int(rc.fold         == r.fold)
print(f"  Violations patient : {viol_pid}  (doit être 0)")
print(f"  Violations groupe  : {viol_grp}  (doit être 0)")
print(f"  Violations pli     : {viol_fold}  (doit être 0)")
assert viol_pid == 0 and viol_grp == 0 and viol_fold == 0, "ERREUR : violation de l'exclusion détectée !"
print("  ✓ Exclusion triple vérifiée")

# ── ACR BI-RADS Guidelines corpus (hardcodé — voir note ci-dessous) ──
# NOTE : Ces 9 entrées sont des résumés éditoriaux du lexique ACR BI-RADS 5e édition.
# Elles ne constituent PAS des documents officiels authentifiés. Ils sont utilisés
# à titre de guide structurel uniquement. Une intégration des PDF officiels
# est recommandée pour un usage clinique réel.
ACR_CORPUS = [
    {"id":"ACR_MAMMO_DESC",  "source":"ACR BI-RADS Atlas 5e éd. — Descripteurs Mammographie",
     "topic":"Forme, Marges et Densité des Masses",
     "text":("Mammographic masses: shape (oval/round/irregular), margins (circumscribed/obscured/"
             "microlobulated/indistinct/spiculated), density (high/equal/low/fat). Spiculated or "
             "indistinct margins in irregular high-density masses carry highest PPV for invasive carcinoma."),
     "recommendation":"Tissue biopsy for spiculated/indistinct irregular masses."},
    {"id":"ACR_MAMMO_CALC",  "source":"ACR BI-RADS Atlas 5e éd. — Calcifications",
     "topic":"Calcifications Suspectes et Bénignes",
     "text":("Benign: skin, vascular, coarse/popcorn, large rod-like. Suspicious: amorphous, coarse "
             "heterogeneous, fine pleomorphic, fine linear/branching — especially grouped, linear "
             "or segmental distributions."),
     "recommendation":"Stereotactic vacuum-assisted biopsy for suspicious morphology calcifications."},
    {"id":"ACR_US_DESC",     "source":"ACR BI-RADS Atlas 5e éd. — Descripteurs Échographie",
     "topic":"Morphologie et Orientation des Lésions Échographiques",
     "text":("US malignancy features: irregular shape, non-parallel orientation (taller-than-wide), "
             "non-circumscribed margins (angular/microlobulated/spiculated), markedly hypoechoic, "
             "posterior acoustic shadowing. Anechoic round circumscribed masses with enhancement = benign cyst."),
     "recommendation":"US-guided core needle biopsy for solid non-circumscribed hypoechoic masses."},
    {"id":"ACR_DENSITY",     "source":"ACR BI-RADS Atlas 5e éd. — Composition Mammaire",
     "topic":"Densité Parenchymateuse et Risque de Masquage",
     "text":("Composition A-D: A=almost entirely fatty, B=scattered fibroglandular, C=heterogeneously "
             "dense, D=extremely dense. ACR C/D significantly reduces mammographic sensitivity → "
             "adjunctive ultrasound warranted."),
     "recommendation":"Adjunctive breast ultrasound for ACR C/D dense breasts."},
    {"id":"ACR_CAT_1_2",     "source":"ACR BI-RADS Atlas 5e éd. — Catégories 1 et 2",
     "topic":"Examens Négatifs et Bénins",
     "text":("BI-RADS 1: completely negative, 0% PPM. BI-RADS 2: definite benign (simple cyst, stable "
             "intramammary lymph node, vascular calcification), 0% PPM. Both: routine screening intervals, "
             "no short-term recall."),
     "recommendation":"Routine standard screening."},
    {"id":"ACR_CAT_3",       "source":"ACR BI-RADS Atlas 5e éd. — Catégorie 3",
     "topic":"Lésion Probablement Bénigne",
     "text":("BI-RADS 3: >0% to ≤2% PPM. Classic: non-palpable circumscribed solid mass on baseline, "
             "focal asymmetry, round calcifications. Management: short-interval follow-up "
             "(unilateral 6 months, bilateral at 12 and 24 months)."),
     "recommendation":"6-month short-interval unilateral follow-up."},
    {"id":"ACR_CAT_4",       "source":"ACR BI-RADS Atlas 5e éd. — Catégorie 4 (4A/4B/4C)",
     "topic":"Anomalie Suspecte nécessitant Biopsie",
     "text":("BI-RADS 4: suspicious, biopsy required. 4A: >2%-≤10% (low suspicion, partially circumscribed "
             "solid). 4B: >10%-≤50% (moderate). 4C: >50%-<95% (high, irregular indistinct solid mass "
             "or fine pleomorphic calcifications)."),
     "recommendation":"Histopathological diagnosis via image-guided core needle biopsy."},
    {"id":"ACR_CAT_5",       "source":"ACR BI-RADS Atlas 5e éd. — Catégorie 5",
     "topic":"Très Fortement Suspect de Malignité",
     "text":("BI-RADS 5: ≥95% PPM. Typical: irregular spiculated high-density mass ± suspicious "
             "microcalcifications; non-parallel hypoechoic mass with acoustic shadowing. "
             "Biopsy mandatory; oncologic surgical planning initiated."),
     "recommendation":"Immediate core needle biopsy + surgical oncology consultation."},
    {"id":"ACR_DISCORDANCE", "source":"ACR Appropriateness Criteria — Corrélation Multimodale",
     "topic":"Arbitrage en cas de Discordance entre Modalités",
     "text":("Discordant mammo/US assessments: management governed by HIGHEST BI-RADS category. "
             "A suspicious finding on either modality warrants targeted tissue sampling regardless "
             "of the other modality's benign assessment."),
     "recommendation":"Highest-suspicion modality governs; perform targeted biopsy."},
]

df_guide = pd.DataFrame(ACR_CORPUS)
E_guide = emb_model.encode(
    df_guide["text"].tolist(), batch_size=16,
    normalize_embeddings=True, show_progress_bar=False
).astype("float32")
idx_guide = faiss.IndexFlatIP(E_guide.shape[1])
idx_guide.add(E_guide)
faiss.write_index(idx_guide, str(WORK_DIR / "acr_guidelines_index.faiss"))
df_guide.to_csv(str(WORK_DIR / "acr_guidelines_chunks.csv"), index=False)
print(f"\nGuides ACR indexés : {idx_guide.ntotal} règles")
print(f"[NOTE] Ces résumés ACR sont à usage de prototype uniquement — non authentifiés officiellement.")

def retrieve_guidelines(query_text, k=2):
    """Recherche dans les guidelines ACR avec traçabilité complète."""
    qvec = emb_model.encode([query_text], normalize_embeddings=True).astype("float32")
    sc, ix = idx_guide.search(qvec, k)
    out = []
    for s, i in zip(sc[0], ix[0]):
        row = df_guide.iloc[i]
        out.append(dict(
            id            = row["id"],
            source        = row["source"],
            topic         = row["topic"],
            score         = round(float(s), 3),
            passage       = row["text"],
            recommendation= row["recommendation"]
        ))
    return out

def evidence(pid, k=3):
    """Recherche de cas similaires pour un patient, avec exclusion stricte."""
    # Obtenir les infos du patient courant
    mask = C.pseudo_id == pid
    if not mask.any():
        return []
    qi = C.index[mask & (C.chunk_idx == 0)]
    if len(qi) == 0:
        qi = C.index[mask]
    qi = qi[0]
    r_row = rep[rep.pseudo_id == pid]
    p_group = r_row["group_pseudo"].iloc[0] if len(r_row) > 0 else None
    p_fold  = int(r_row["fold"].iloc[0]) if len(r_row) > 0 else -1
    qv = E[qi]
    return search_cases(qv, pid, p_group, p_fold, k=k)

print("\nTest evidence() :")
_test_pid = rep.iloc[0].pseudo_id
_ev = evidence(_test_pid, k=2)
print(f"  Patient {_test_pid} → {len(_ev)} cas similaires (exclus: même patient/groupe/pli)")
for e in _ev:
    print(f"    {e['source_patient']} score={e['score']}")

=== Test de sécurité : exclusion patient / groupe / pli ===
  Violations patient : 0  (doit être 0)
  Violations groupe  : 0  (doit être 0)
  Violations pli     : 0  (doit être 0)
  ✓ Exclusion triple vérifiée

Guides ACR indexés : 9 règles
[NOTE] Ces résumés ACR sont à usage de prototype uniquement — non authentifiés officiellement.

Test evidence() :
  Patient P0001 → 2 cas similaires (exclus: même patient/groupe/pli)
    P0076 score=0.937
    P0015 score=0.936


## R3 : Dossiers patients — isolation stricte des modalités

In [7]:
# ============================================================
# R3 : Construction des dossiers patients pour l'agent RAG
# Isolation STRICTE des modalités (zéro fuite)
# ============================================================

rep_i = rep.set_index("pseudo_id")

def clean_passage(s):
    return re.sub(r"^[\s:.]+", "", s)

def build_dossier(pid):
    """
    Construit le dossier d'un patient avec :
    - Prédictions vision BRUTES (sans modification)
    - Guides sélectionnés par condition (zéro fuite croisée)
    - Cas similaires avec exclusion triple
    - Valeurs manquantes → null + motif (jamais fabriquées)
    """
    d = dict(pseudo_id=pid)

    # ── 1. Prédictions Vision (telles que reçues, sans modification) ──
    if pid in pred.index:
        v = pred.loc[pid]
        p_norm = float(v.get("p_normal_mg", np.nan))
        p_ben  = float(v.get("p_benign_mg",  np.nan))
        p_mal  = float(v.get("p_malignant_mg", np.nan))
        p_ent  = float(v.get("entropie_mg",   np.nan))
        p_us   = None if pd.isna(v.get("p_malignant_us")) else float(v.get("p_malignant_us"))
        p_fus  = None if pd.isna(v.get("p_malignant_fusion")) else float(v.get("p_malignant_fusion"))
        pb_raw = v.get("predicted_birads", None)
        cf_raw = v.get("confidence", None)
        # Valider predicted_birads
        valid_birads = {"1","2","3","4A","4B","4C","5","0"}
        pb_str = str(pb_raw) if pd.notna(pb_raw) and str(pb_raw) in valid_birads else None
        cf_val = round(float(cf_raw), 3) if pd.notna(cf_raw) else None

        d["vision"] = dict(
            predicted_birads   = pb_str,
            predicted_birads_missing_reason = None if pb_str else f"Valeur brute '{pb_raw}' invalide ou absente",
            confidence         = cf_val,
            confidence_note    = ("Valeur brute du modèle, non calibrée cliniquement"
                                  if cf_val is not None else None),
            p_normal_mg        = round(p_norm, 3) if not np.isnan(p_norm) else None,
            p_benign_mg        = round(p_ben,  3) if not np.isnan(p_ben)  else None,
            p_malignant_mg     = round(p_mal,  3) if not np.isnan(p_mal)  else None,
            entropy_mg         = round(p_ent,  3) if not np.isnan(p_ent)  else None,
            p_malignant_us     = round(p_us,   3) if p_us  is not None else None,
            p_malignant_fusion = round(p_fus,  3) if p_fus is not None else None,
            n_images_mammo     = int(v.get("n_img_mg", 0)) if pd.notna(v.get("n_img_mg")) else None,
            standard_device    = bool(v.get("appareil_standard", True)),
            provenance         = "predictions_patient.csv — Vision Agent (Box 3) — NON validé cliniquement"
        )
    else:
        d["vision"] = None

    # ── 2. Heatmaps XAI ──
    d["heatmap"] = fiches.get(pid)  # None si absent

    # ── 3. Rapport texte (findings uniquement) ──
    has_rep = pid in rep_i.index
    if has_rep:
        raw_txt = rep_i.loc[pid, "text"]
        d["findings"] = clean_passage(raw_txt)[:1500]
    else:
        d["findings"] = None  # Explicitement null — jamais fabriqué

    # ── 4. Guides ACR par condition (ISOLATION STRICTE) ──

    # Condition IMAGE : requête basée UNIQUEMENT sur les prédictions vision (jamais sur le texte)
    if d["vision"] and d["vision"]["predicted_birads"]:
        pb = d["vision"]["predicted_birads"]
        pm = d["vision"]["p_malignant_fusion"] or d["vision"]["p_malignant_mg"] or 0.0
        query_img = (f"mammography imaging findings suspicion BI-RADS {pb} "
                     f"malignant probability {pm:.2f}")
    elif d["vision"]:
        pm = d["vision"]["p_malignant_mg"] or 0.0
        query_img = f"mammography imaging findings malignant probability {pm:.2f}"
    else:
        query_img = "mammography imaging findings breast evaluation"
    d["guidelines_image"] = retrieve_guidelines(query_img, k=2)

    # Condition TEXTE : requête basée UNIQUEMENT sur le texte (jamais sur les scores vision)
    if d["findings"]:
        query_txt = d["findings"][:200] + " breast density morphology lesion findings"
        d["guidelines_texte"] = retrieve_guidelines(query_txt, k=2)
    else:
        d["guidelines_texte"] = []

    # Condition MULTIMODALE : combine les deux (disponible seulement en condition les_deux)
    query_both = ""
    if d["findings"]:
        query_both += d["findings"][:120] + " "
    if d["vision"] and d["vision"]["predicted_birads"]:
        query_both += f"imaging suspicion BI-RADS {d['vision']['predicted_birads']}"
    else:
        query_both += "breast imaging multimodal evaluation"
    d["guidelines_multimodal"] = retrieve_guidelines(query_both, k=2)

    # ── 5. Cas similaires (texte uniquement, exclusion triple) ──
    d["evidence"] = []
    if has_rep:
        raw_ev = evidence(pid, k=3)
        d["evidence"] = [dict(e, passage=clean_passage(e["passage"])) for e in raw_ev]

    return d

# Construire les dossiers pour le pli 0 (patients avec heatmaps)
if fiches:
    target_pids = sorted(fiches.keys())
else:
    # Fallback : pli 0 du dataset
    target_pids = sorted(rep[rep.fold == 0].pseudo_id.tolist())
    if not target_pids:
        target_pids = sorted(rep.pseudo_id.tolist()[:30])

print(f"Construction des dossiers pour {len(target_pids)} patients...")
D = {}
_errors = []
for pid in target_pids:
    try:
        D[pid] = build_dossier(pid)
    except Exception as e:
        _errors.append((pid, str(e)))
        print(f"  [ERREUR] {pid} : {e}")

print(f"\nDossiers construits : {len(D)} patients")
print(f"  avec vision        : {sum(d['vision'] is not None for d in D.values())}")
print(f"  avec rapport texte : {sum(d['findings'] is not None for d in D.values())}")
print(f"  avec heatmap XAI   : {sum(d['heatmap'] is not None for d in D.values())}")
print(f"  avec guides image  : {sum(len(d['guidelines_image'])>0 for d in D.values())}")
print(f"  avec guides texte  : {sum(len(d['guidelines_texte'])>0 for d in D.values())}")
print(f"  avec cas similaires: {sum(len(d['evidence'])>0 for d in D.values())}")
if _errors:
    print(f"  Erreurs : {_errors}")

# Sauvegarde
with open(str(WORK_DIR / "dossiers_patients.jsonl"), "w", encoding="utf-8") as f:
    for d in D.values():
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
print("Sauvegardé : dossiers_patients.jsonl")

Construction des dossiers pour 64 patients...

Dossiers construits : 64 patients
  avec vision        : 64
  avec rapport texte : 62
  avec heatmap XAI   : 64
  avec guides image  : 64
  avec guides texte  : 62
  avec cas similaires: 62
Sauvegardé : dossiers_patients.jsonl


## R4 : Schéma de rapport, prompts, vérification d'isolation mathématique

In [8]:
# ============================================================
# R4 : Schéma de rapport, construction des prompts, vérification d'isolation
# ============================================================

# ── SYSTÈME de l'agent ──
SYSTEM_BASE = """You are the BCMID Multimodal Medical AI Reasoning Agent.
You synthesize deep learning vision outputs, clinical findings, and ACR BI-RADS guidelines
to assist radiologists. You generate structured research reports.

STRICT RULES:
1. Never invent probabilities, BI-RADS categories, or findings not provided.
2. Cite guidelines as [G1], [G2] and similar cases as [E1], [E2].
3. State concrete, evidence-based recommendations.
4. Always note that models are research baselines, not clinically validated.
5. NEVER claim confirmed diagnosis. ALWAYS mandate human radiologist review.
6. Answer with ONE valid JSON object only — matching the OUTPUT SCHEMA exactly.
"""

SYSTEM_EXTRA = """
Additional constraints:
- Evidence passages (E1, E2) are from OTHER patients' reports, not this patient.
- Ultrasound model is secondary; mammography carries primary weight.
- These are research baseline models, not clinically validated autonomous tools.
- Missing values must be reported as null with a reason, never fabricated.
"""

# ── Schéma officiel du rapport structuré ──
SCHEMA = {
    "patient_summary": {
        "pseudo_id"         : "str",
        "condition"         : "str: image | texte | les_deux",
        "acquisition_device": "str: Standard | Non-standard (reduced resolution)"
    },
    "predicted_birads"    : "str or null: 1/2/3/4A/4B/4C/5 — null si condition texte",
    "confidence"          : "float or null: [0.0, 1.0] — null si condition texte",
    "interpretation"      : "str: max 100 mots, basé uniquement sur les données fournies",
    "supporting_evidence" : "list[str]: citations [G1] [G2] [E1] [E2]",
    "visual_explanations" : "str or null: résumé heatmaps réelles — null si condition texte",
    "recommendations"     : "list[str]: actions cliniques concrètes",
    "limitations"         : "list[str]: incertitudes, qualité appareil, avertissement modèle",
    "human_review_required": True
}
SCHEMA_KEYS = list(SCHEMA.keys())
print("Clés du schéma :", SCHEMA_KEYS)

def zone_desc(z):
    """Convertit une zone heatmap en description textuelle sans invention."""
    if isinstance(z, str):
        return z
    if isinstance(z, dict):
        pos = z.get("position", "")
        if pos:
            return str(pos)
        # Calcul à partir de coordonnées si disponibles
        if "y" in z and "x" in z:
            ph = "upper" if z["y"] < 0.33 else ("middle" if z["y"] < 0.66 else "lower")
            pw = "medial" if z["x"] < 0.33 else ("central" if z["x"] < 0.66 else "lateral")
            surf = f" (surface {z['x_width']*z['y_height']*100:.0f}%)" if "x_width" in z else ""
            return f"{ph}-{pw}{surf}"
    return str(z)

def build_heatmap_summary(hm):
    """Résume les heatmaps RÉELLES — jamais de description fabriquée."""
    if not hm:
        return None  # Absent → null, jamais de phrase inventée
    mg_maps = hm.get("mammography_heatmaps", [])
    us_maps = hm.get("ultrasound_heatmaps", [])
    out = {}
    if mg_maps:
        out["mammography"] = "; ".join(
            f"View {m.get('vue','MG')}: zones=[" +
            ",".join(zone_desc(z) for z in m.get("zones", [])[:3]) + "]" +
            f" outside_breast={m.get('attention_hors_sein',0):.1%}"
            for m in mg_maps[:3]
        )
    else:
        out["mammography"] = "Grad-CAM heatmap: not available for this patient"
    if us_maps:
        out["ultrasound"] = "; ".join(
            "US zones=[" + ",".join(zone_desc(z) for z in m.get("zones",[])[:2]) + "]"
            for m in us_maps[:2]
        )
    else:
        out["ultrasound"] = "Ultrasound Grad-CAM: not available for this patient"
    return out

def build_prompt2(d, cond):
    """
    Construit le prompt avec ISOLATION ABSOLUE des modalités.
    - 'image'   : vision + heatmaps + guides_image. ZÉRO texte, ZÉRO cas similaires.
    - 'texte'   : rapport + cas similaires + guides_texte. ZÉRO vision, ZÉRO heatmap.
    - 'les_deux': vision + heatmaps + rapport + cas similaires + guides_multimodal.
    Preuves mathématiques d'isolation incluses.
    """
    parts = [f"CONDITION: {cond}", "OUTPUT SCHEMA: " + json.dumps(SCHEMA, ensure_ascii=False)]

    # === A) DONNÉES IMAGE ===
    if cond in ("image", "les_deux"):
        v = d.get("vision")
        if v:
            vis_data = dict(
                mammography_probabilities=dict(
                    normal=v["p_normal_mg"], benign=v["p_benign_mg"], malignant=v["p_malignant_mg"]
                ),
                uncertainty_entropy=v["entropy_mg"],
                n_mammography_images=v["n_images_mammo"],
                standard_device=v["standard_device"],
                ultrasound_malignant_probability=v["p_malignant_us"],
                fusion_malignant_probability=v["p_malignant_fusion"],
                note=v["provenance"]
            )
            parts.append("VISION MODEL SCORES: " + json.dumps(vis_data, ensure_ascii=False))
            # Heatmaps XAI
            hm_summ = build_heatmap_summary(d.get("heatmap"))
            if hm_summ:
                parts.append("HEATMAP_XAI: " + json.dumps(hm_summ, ensure_ascii=False))
            else:
                parts.append("HEATMAP_XAI: null — not available for this patient")
        else:
            parts.append("VISION MODEL SCORES: null — prediction not available for this patient")
            parts.append("HEATMAP_XAI: null — not available for this patient")
        # Guides IMAGE (issus uniquement de vision)
        for k, g in enumerate(d.get("guidelines_image", []), 1):
            parts.append(f"CLINICAL GUIDELINE G{k} [{g['id']} — {g['source']}]: "
                        f"{g['passage'][:300]} (Recommendation: {g['recommendation']})")

    # === B) DONNÉES TEXTE ===
    if cond in ("texte", "les_deux"):
        findings = d.get("findings")
        parts.append("THIS PATIENT'S REPORT FINDINGS (anonymized): " +
                     (findings if findings else "NOT AVAILABLE — report absent"))
        # Cas similaires (exclus de la condition image)
        for k, e in enumerate(d.get("evidence", []), 1):
            parts.append(f"SIMILAR CASE E{k} [Source: {e['source_patient']}, "
                        f"similarity {e['score']}]: {e['passage']}")
        # Guides TEXTE (issus uniquement du texte)
        gl_key = "guidelines_texte" if cond == "texte" else "guidelines_multimodal"
        for k, g in enumerate(d.get(gl_key, []), 1):
            parts.append(f"CLINICAL GUIDELINE G{k} [{g['id']} — {g['source']}]: "
                        f"{g['passage'][:300]} (Recommendation: {g['recommendation']})")

    return SYSTEM_BASE + SYSTEM_EXTRA, "\n".join(parts)

# ── Construction des prompts pour tous les patients ──
print("Construction des prompts...")
pm_rows = []
for pid, d in D.items():
    has_rep  = d["findings"] is not None
    has_vis  = d["vision"] is not None
    r_row    = rep[rep.pseudo_id == pid]
    tctx = False
    if len(r_row) > 0 and "treatment_ctx" in rep.columns:
        tctx = bool(r_row["treatment_ctx"].iloc[0])

    for cond in ("image", "texte", "les_deux"):
        if cond == "texte" and not has_rep:
            continue  # Condition texte impossible sans rapport
        sys_p, usr_p = build_prompt2(d, cond)
        n_ev = len(d["evidence"]) if cond != "image" else 0
        n_gd = len(d.get(
            "guidelines_image" if cond == "image" else
            ("guidelines_texte" if cond == "texte" else "guidelines_multimodal"), []
        ))
        pm_rows.append(dict(
            pseudo_id     = pid,
            condition     = cond,
            system        = sys_p,
            user          = usr_p,
            n_evidence    = n_ev,
            n_guidelines  = n_gd,
            has_vision    = has_vis,
            has_report    = has_rep,
            standard_device = bool(d["vision"]["standard_device"]) if has_vis else True,
            treatment_ctx = tctx
        ))

Pm = pd.DataFrame(pm_rows)
print(f"Prompts générés : {len(Pm)} | par condition : {Pm.condition.value_counts().to_dict()}")

# ── VÉRIFICATION MATHÉMATIQUE D'ISOLATION ──
leak_txt_in_img = int(Pm[Pm.condition=="image"].user.str.contains(
    "THIS PATIENT'S REPORT", regex=False).sum())
leak_vis_in_txt = int(Pm[Pm.condition=="texte"].user.str.contains(
    "VISION MODEL SCORES", regex=False).sum())
leak_ev_in_img  = int(Pm[Pm.condition=="image"].user.str.contains(
    "SIMILAR CASE E", regex=False).sum())

print(f"\n── Vérification d'isolation ──")
print(f"  Fuite texte dans condition image    : {leak_txt_in_img}  (doit être 0)")
print(f"  Fuite vision dans condition texte   : {leak_vis_in_txt}  (doit être 0)")
print(f"  Fuite cas similaires dans image     : {leak_ev_in_img}  (doit être 0)")
assert leak_txt_in_img == 0, "ERREUR : fuite texte dans condition image !"
assert leak_vis_in_txt == 0, "ERREUR : fuite vision dans condition texte !"
assert leak_ev_in_img  == 0, "ERREUR : fuite cas similaires dans image !"
print("  ✓ Isolation des modalités vérifiée mathématiquement")

# Sauvegarde
Pm.to_json(str(WORK_DIR / "prompts.jsonl"), orient="records", lines=True, force_ascii=False)
print("Sauvegardé : prompts.jsonl")

Clés du schéma : ['patient_summary', 'predicted_birads', 'confidence', 'interpretation', 'supporting_evidence', 'visual_explanations', 'recommendations', 'limitations', 'human_review_required']
Construction des prompts...
Prompts générés : 190 | par condition : {'image': 64, 'les_deux': 64, 'texte': 62}

── Vérification d'isolation ──
  Fuite texte dans condition image    : 0  (doit être 0)
  Fuite vision dans condition texte   : 0  (doit être 0)
  Fuite cas similaires dans image     : 0  (doit être 0)
  ✓ Isolation des modalités vérifiée mathématiquement
Sauvegardé : prompts.jsonl


## R5 : Agent LLM Qwen2.5-3B (optionnel — CPU/GPU auto-détecté)

In [9]:
# ============================================================
# R5 : Agent LLM — Qwen2.5-3B-Instruct (optionnel GPU/CPU)
# Si aucun GPU, mode extractif automatique.
# ============================================================
import time

LLM_ENABLED = False   # Sera mis à True si le modèle se charge
model = None; tok = None

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    MID = "Qwen/Qwen2.5-3B-Instruct"
    print(f"Chargement du modèle LLM : {MID}")
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(MID)

    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            MID, dtype=torch.float16, device_map="auto"
        ).eval()
        gpu_name = torch.cuda.get_device_name(0)
        mem_gb   = round(torch.cuda.memory_allocated() / 1e9, 1)
        print(f"  GPU : {gpu_name} | Mémoire : {mem_gb} Go | Chargé en {time.time()-t0:.0f} s")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MID, dtype=torch.float32
        ).eval()
        print(f"  Mode CPU (pas de GPU détecté) | Chargé en {time.time()-t0:.0f} s")
        print("  [AVERTISSEMENT] Génération LLM sur CPU sera très lente (>5 min/cas)")

    LLM_ENABLED = True
    print("✓ LLM prêt")

except ImportError:
    print("[INFO] transformers/torch non disponibles — mode extractif uniquement")
except Exception as e:
    print(f"[INFO] LLM non chargé ({e}) — mode extractif")

def generate_llm(system, user, max_new=450):
    """Génération LLM avec validation de sortie."""
    if not LLM_ENABLED or model is None:
        return None
    msgs = [{"role":"system","content":system}, {"role":"user","content":user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def extract_json(txt):
    """Extrait le premier objet JSON valide d'un texte brut."""
    if txt is None:
        return None
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

print(f"\nMode LLM : {'ACTIF (Qwen2.5-3B)' if LLM_ENABLED else 'INACTIF — mode extractif'}")

Chargement du modèle LLM : Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  GPU : Tesla T4 | Mémoire : 3.1 Go | Chargé en 40 s
✓ LLM prêt

Mode LLM : ACTIF (Qwen2.5-3B)


## R6 : Validateur exhaustif + 10 tests adversariaux

In [10]:
# ============================================================
# R6 : Validateur exhaustif + 10 tests adversariaux
# ============================================================

FORBIDDEN_PATTERNS = [
    r"bi-?\s?rads?\s*\(?\s*[0-6]",           # Chiffre BI-RADS dans le texte libre
    r"\bhas\s+(breast\s+)?cancer\b",          # Diagnostic affirmé
    r"\bdiagnos(is|ed)\s+(of|as|with)\b",     # Diagnostic posé
    r"\bis\s+(definitely|certainly)\s+(malignant|benign)\b",
    r"\bconfirmed\s+(malignant|benign|cancer)\b",
    r"\b(calibrated|clinically\s+validated)\s+confidence\b"  # Confiance auto-déclarée calibrée
]

def validate_medical_report(rep_dict, row, F, report_text=""):
    """
    Validateur exhaustif du rapport médical structuré.

    Contrôles effectués :
    1. Structure complète (tous les champs SCHEMA présents)
    2. Isolation des modalités (zéro fuite)
    3. BI-RADS et confiance (valides, cohérents avec la condition)
    4. Interprétation (longueur, formulations interdites, nombres)
    5. XAI (heatmaps réelles, pas de phrases fabriquées)
    6. Citations (sources réellement fournies)
    7. Discordance mammo/écho signalée
    8. Avertissement modèle de recherche présent
    9. Appareil non-standard signalé

    Retourne : list[str] d'erreurs (vide = rapport valide techniquement)
    Statuts : "checks_passed" | "review_required" | "not_evaluated"
    """
    errs = []

    if rep_dict is None or not isinstance(rep_dict, dict):
        return ["Rapport invalide (non-dict ou vide)"]

    # ── 1. Champs obligatoires ──
    for k in SCHEMA_KEYS:
        if k not in rep_dict:
            errs.append(f"champ_manquant:{k}")

    # ── 2. patient_summary (identité et condition) ──
    ps = rep_dict.get("patient_summary", {})
    if isinstance(ps, dict):
        if ps.get("pseudo_id") != row.pseudo_id:
            errs.append(f"pseudo_id_divergent:{ps.get('pseudo_id')} vs attendu:{row.pseudo_id}")
        if ps.get("condition") != row.condition:
            errs.append(f"condition_divergente:{ps.get('condition')} vs attendu:{row.condition}")

    # ── 3. predicted_birads et confidence ──
    pb = rep_dict.get("predicted_birads")
    cf = rep_dict.get("confidence")
    VALID_BIRADS = {"1","2","3","4A","4B","4C","5"}
    if row.condition == "texte":
        if pb is not None:
            errs.append("fuite_vision:predicted_birads fourni en condition texte")
        if cf is not None:
            errs.append("fuite_vision:confidence fournie en condition texte")
    else:
        if pb is not None and str(pb) not in VALID_BIRADS:
            errs.append(f"birads_invalide:{pb}")
        if cf is not None:
            try:
                cf_f = float(cf)
                if not (0.0 <= cf_f <= 1.0):
                    errs.append(f"confidence_hors_bornes:{cf}")
            except (TypeError, ValueError):
                errs.append(f"confidence_non_numerique:{cf}")

    # ── 4. Interprétation ──
    interp = str(rep_dict.get("interpretation", ""))
    if not interp.strip():
        errs.append("interpretation_vide")
    if len(interp.split()) > 100:
        errs.append(f"interpretation_trop_longue:{len(interp.split())} mots")

    for p in FORBIDDEN_PATTERNS:
        if re.search(p, interp, re.I):
            errs.append(f"formulation_interdite:{p[:30]}")

    # Vérification des nombres (mesures légitimes du rapport vs probabilités modèle)
    model_vals = []
    if F.get("mammography_probabilities"):
        model_vals += list(F["mammography_probabilities"].values())
    if F.get("uncertainty_entropy") is not None:
        model_vals.append(F["uncertainty_entropy"])
    if F.get("ultrasound_malignant_probability") is not None:
        model_vals.append(F["ultrasound_malignant_probability"])

    report_numbers = set(re.findall(r"\d+(?:\.\d+)?", report_text)) if report_text else set()

    interp_clean = re.sub(r"\[[EG]\d+\]", "", interp)
    for s in re.findall(r"\d+(?:\.\d+)?", interp_clean):
        if s in report_numbers:
            continue  # Mesure légitime du rapport (ex: 10mm, 45 ans)
        if any(abs(float(s) - v) < 0.006 or abs(float(s) - 100*v) < 0.6
               for v in model_vals if v is not None):
            continue  # Probabilité du modèle reconnue
        # Nombre inconnu → potentiellement inventé
        errs.append(f"nombre_externe_suspect:{s}")

    # Cohérence incertitude
    if re.search(r"low\s+(entropy|uncertainty)", interp, re.I):
        ent = F.get("uncertainty_entropy", 0)
        if ent is not None and ent >= 0.5:
            errs.append("incertitude_elevee_decrite_comme_faible")

    # ── 5. Explications visuelles XAI ──
    vx = rep_dict.get("visual_explanations")
    if row.condition == "texte":
        if vx is not None:
            errs.append("fuite_vision:visual_explanations en condition texte")
    else:
        if vx is None:
            errs.append("visual_explanations_manquant_en_condition_image")
        elif isinstance(vx, str):
            # Vérifier phrases génériques inventées
            generic = [
                "attention focused on breast parenchymal density and focal asymmetry",
                "significant enhancement in the upper outer quadrant"
            ]
            for g in generic:
                if g.lower() in vx.lower():
                    errs.append(f"xai_phrase_inventee:{g[:40]}")

    # ── 6. Citations (vérifier que les références existent) ──
    n_ev = int(row.n_evidence) if hasattr(row, "n_evidence") else 0
    n_gd = int(row.n_guidelines) if hasattr(row, "n_guidelines") else 2
    allowed_e = {f"E{k}" for k in range(1, n_ev+1)}
    allowed_g = {f"G{k}" for k in range(1, n_gd+1)}
    allowed_refs = allowed_e | allowed_g

    all_text = json.dumps(rep_dict, ensure_ascii=False)
    for ref in re.findall(r"\[([EG]\d+)\]", all_text):
        if ref not in allowed_refs:
            errs.append(f"citation_inexistante:{ref}")

    # Cas similaires en condition image → fuite
    similar = rep_dict.get("similar_cases", [])
    if row.condition == "image" and similar:
        errs.append("fuite:cas_similaires_en_condition_image")

    # ── 7. Discordance mammo/écho ──
    pm = (F.get("mammography_probabilities") or {}).get("malignant", 0.0) or 0.0
    pu = F.get("ultrasound_malignant_probability")
    if pu is not None and ((pm >= 0.5) != (pu >= 0.5)):
        # Discordance : doit être mentionnée
        if not re.search(r"different direction|discord|disagree|diverg|contradict",
                         all_text, re.I):
            errs.append("discordance_mammo_echo_non_signalee")

    # Inversion mammo/écho dans le rapport (ex: valeurs échangées)
    if pu is not None and F.get("mammography_probabilities"):
        # Vérifier que les valeurs ne sont pas inversées dans le texte
        pm_str = f"{pm:.2f}"; pu_str = f"{pu:.2f}"
        # Si le rapport cite pm comme valeur échographie et vice-versa → erreur
        # (Contrôle heuristique — non exhaustif)
        if (f"ultrasound {pm_str}" in all_text and f"mammograph {pu_str}" in all_text and
            pm_str != pu_str):
            errs.append("inversion_scores_mammo_echo_suspectee")

    # ── 8. Recommandations et limitations ──
    recs = rep_dict.get("recommendations", [])
    if not recs:
        errs.append("recommandations_vides")
    lims = rep_dict.get("limitations", [])
    if not lims:
        errs.append("limitations_vides")
    if not any("baseline" in str(l).lower() or "research" in str(l).lower() or
               "not clinically" in str(l).lower() for l in lims):
        errs.append("avertissement_modele_absent_des_limitations")

    # ── 9. Appareil non standard ──
    if not F.get("standard_device", True):
        if not re.search(r"less reliable|non-standard|nonstandard|reduced resolution", all_text, re.I):
            errs.append("appareil_non_standard_non_signale")

    return errs


# ─────────────────────────────────────────────────────────────────────────────
# 10 TESTS ADVERSARIAUX OBLIGATOIRES
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("BATTERIE DE 10 TESTS ADVERSARIAUX")
print("=" * 65)

class FakeRow:
    """Ligne de prompt simulée pour les tests."""
    def __init__(self, pid="P_TEST", cond="les_deux", std=True, n_ev=2, n_gd=2, has_rep=True):
        self.pseudo_id     = pid
        self.condition     = cond
        self.standard_device = std
        self.n_evidence    = n_ev
        self.n_guidelines  = n_gd
        self.has_report    = has_rep
        self.user = "VISION MODEL SCORES: {}\nTHIS PATIENT'S REPORT: Normal tissue 10mm lesion."

RESULTS = {}

# ── T1 : Inversion des scores mammo/écho signalée ──
F_t1 = dict(
    mammography_probabilities=dict(normal=0.70, benign=0.16, malignant=0.14),
    ultrasound_malignant_probability=0.85,
    uncertainty_entropy=0.88, standard_device=True
)
rep_t1_ok = dict(
    patient_summary=dict(pseudo_id="P_TEST", condition="les_deux", acquisition_device="Standard"),
    predicted_birads="4B", confidence=0.72,
    interpretation=("Mammography malignant 0.14, benign 0.16. Ultrasound malignant 0.85. "
                    "The two modality scores point in different directions — ultrasound dominates."),
    supporting_evidence=["[G1] ACR: higher category governs"],
    visual_explanations="Mammography Grad-CAM: not available for this patient",
    recommendations=["Core needle biopsy per ACR (ultrasound finding governs)."],
    limitations=["Research baseline model, not clinically validated."],
    human_review_required=True
)
errs = validate_medical_report(rep_t1_ok, FakeRow(), F_t1, report_text="10mm lesion")
RESULTS["T1_inversion_signalee"] = {"status": "PASS" if not errs else "FAIL", "errors": errs}
print(f"T1 Inversion scores signalée : {'✓ PASS' if not errs else '✗ FAIL'} {errs}")

# ── T2 : Inversion non signalée (doit échouer) ──
rep_t2_bad = dict(rep_t1_ok)
rep_t2_bad["interpretation"] = "Mammography 0.14. Ultrasound 0.85."
errs = validate_medical_report(rep_t2_bad, FakeRow(), F_t1)
ok = any("discordance" in e for e in errs)
RESULTS["T2_inversion_non_signalee"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T2 Inversion non signalée détectée : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T3 : Fausse citation (G5 n'existe pas avec n_gd=2) ──
rep_t3 = dict(rep_t1_ok)
rep_t3["supporting_evidence"] = ["[G5] ACR Atlas: imaginary reference"]
errs = validate_medical_report(rep_t3, FakeRow(n_gd=2), F_t1)
ok = any("citation_inexistante" in e or "G5" in e for e in errs)
RESULTS["T3_fausse_citation"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T3 Fausse citation G5 détectée : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T4 : Passage inventé → nombre externe non reconnu ──
rep_t4 = dict(rep_t1_ok)
rep_t4["interpretation"] = "Patient probability is 0.42."  # 0.42 n'est pas dans F_t1
errs = validate_medical_report(rep_t4, FakeRow(), F_t1)
ok = any("nombre_externe_suspect" in e and "0.42" in e for e in errs)
RESULTS["T4_nombre_invente"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T4 Nombre externe 0.42 détecté : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T5 : Mesure légitime du rapport (10mm) ne doit PAS être rejetée ──
rep_t5 = dict(rep_t1_ok)
rep_t5["interpretation"] = "A 10mm hypoechoic lesion with 0.14 malignant probability on mammography."
# report_text contient "10mm" → doit être autorisé
errs = validate_medical_report(rep_t5, FakeRow(), F_t1, report_text="10mm lesion")
# 10mm acceptable, 0.14 aussi (dans F_t1)
ok = not any("10" in e for e in errs)
RESULTS["T5_mesure_legitime"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T5 Mesure 10mm légitime acceptée : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T6 : Texte seul sans vision (fuite vision doit être détectée) ──
F_t6 = dict(mammography_probabilities=None, ultrasound_malignant_probability=None,
            uncertainty_entropy=None, standard_device=True)
rep_t6_bad = dict(
    patient_summary=dict(pseudo_id="P_TEST", condition="texte", acquisition_device="Standard"),
    predicted_birads="3", confidence=0.65,  # Ces champs ne doivent pas être là
    interpretation="Text-only evaluation.",
    supporting_evidence=["[G1] ACR density"],
    visual_explanations="Heatmap shows upper zone.",  # Ne doit pas être là
    recommendations=["Follow-up."],
    limitations=["Research baseline model."],
    human_review_required=True
)
errs = validate_medical_report(rep_t6_bad, FakeRow(cond="texte", n_ev=0), F_t6)
ok = (any("fuite_vision:predicted_birads" in e or "fuite_vision" in e for e in errs) and
      any("visual_explanations" in e for e in errs))
RESULTS["T6_texte_fuite_vision"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T6 Fuite vision en condition texte détectée : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T7 : Image seule — cas similaires interdits ──
F_t7 = dict(mammography_probabilities=dict(normal=0.2,benign=0.3,malignant=0.5),
            ultrasound_malignant_probability=None,
            uncertainty_entropy=0.3, standard_device=True)
rep_t7_bad = dict(
    patient_summary=dict(pseudo_id="P_TEST", condition="image", acquisition_device="Standard"),
    predicted_birads="4A", confidence=0.50,
    interpretation="Mammography malignant 0.50.",
    supporting_evidence=["[G1] ACR dense"],
    similar_cases=[{"id": "E1", "passage": "similar benign patient..."}],  # INTERDIT
    visual_explanations="Mammography Grad-CAM: not available for this patient",
    recommendations=["Biopsy."],
    limitations=["Research baseline model, not clinically validated."],
    human_review_required=True
)
errs = validate_medical_report(rep_t7_bad, FakeRow(cond="image", n_ev=0), F_t7)
ok = any("cas_similaires" in e or "fuite" in e for e in errs)
RESULTS["T7_image_cas_similaires_interdits"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T7 Cas similaires en condition image détectés : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T8 : Prédiction manquante (pred_birads=None autorisé) ──
F_t8 = dict(mammography_probabilities=None, ultrasound_malignant_probability=None,
            uncertainty_entropy=None, standard_device=True)
rep_t8_ok = dict(
    patient_summary=dict(pseudo_id="P_TEST", condition="image", acquisition_device="Standard"),
    predicted_birads=None, confidence=None,
    interpretation="No vision model prediction available for this patient.",
    supporting_evidence=[],
    visual_explanations="Mammography Grad-CAM: not available for this patient",
    recommendations=["Human radiologist review required."],
    limitations=["Research baseline model, not clinically validated. Prediction absent."],
    human_review_required=True
)
errs = validate_medical_report(rep_t8_ok, FakeRow(cond="image", n_ev=0), F_t8)
ok = not any("birads" in e.lower() or "confidence" in e.lower() for e in errs)
RESULTS["T8_prediction_manquante"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T8 Prédiction manquante (null) acceptée : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── T9 : Sortie LLM invalide (JSON malformé → None) ──
raw_invalid = "The patient has... [incomplete json"
extracted = extract_json(raw_invalid)
ok = extracted is None
RESULTS["T9_sortie_llm_invalide"] = {"status": "PASS" if ok else "FAIL", "extracted": extracted}
print(f"T9 JSON LLM invalide → None : {'✓ PASS' if ok else '✗ FAIL'} extrait={extracted}")

# ── T10 : Appareil non standard non signalé (doit échouer) ──
F_t10 = dict(mammography_probabilities=dict(normal=0.3,benign=0.3,malignant=0.4),
             ultrasound_malignant_probability=None,
             uncertainty_entropy=0.4, standard_device=False)  # Appareil NON STANDARD
rep_t10_bad = dict(
    patient_summary=dict(pseudo_id="P_TEST", condition="image", acquisition_device="Standard"),
    predicted_birads="3", confidence=0.60,
    interpretation="Mammography malignant 0.40.",  # Ne mentionne PAS l'appareil non standard
    supporting_evidence=["[G1] ACR"],
    visual_explanations="Mammography Grad-CAM: not available",
    recommendations=["Follow-up."],
    limitations=["Research baseline model, not clinically validated."],
    human_review_required=True
)
errs = validate_medical_report(rep_t10_bad, FakeRow(cond="image", std=False), F_t10)
ok = any("non_standard" in e or "appareil" in e for e in errs)
RESULTS["T10_appareil_non_standard"] = {"status": "PASS" if ok else "FAIL", "errors": errs}
print(f"T10 Appareil non standard non signalé détecté : {'✓ PASS' if ok else '✗ FAIL'} {errs}")

# ── Résumé ──
print("\n" + "=" * 65)
n_pass = sum(1 for v in RESULTS.values() if v["status"] == "PASS")
print(f"RÉSULTAT : {n_pass}/{len(RESULTS)} tests passés")
for name, r in RESULTS.items():
    icon = "✓" if r["status"] == "PASS" else "✗"
    print(f"  {icon} {name}")
print("=" * 65)

BATTERIE DE 10 TESTS ADVERSARIAUX
T1 Inversion scores signalée : ✓ PASS []
T2 Inversion non signalée détectée : ✓ PASS ['discordance_mammo_echo_non_signalee']
T3 Fausse citation G5 détectée : ✓ PASS ['citation_inexistante:G5']
T4 Nombre externe 0.42 détecté : ✓ PASS ['nombre_externe_suspect:0.42', 'discordance_mammo_echo_non_signalee']
T5 Mesure 10mm légitime acceptée : ✓ PASS ['discordance_mammo_echo_non_signalee']
T6 Fuite vision en condition texte détectée : ✓ PASS ['fuite_vision:predicted_birads fourni en condition texte', 'fuite_vision:confidence fournie en condition texte', 'fuite_vision:visual_explanations en condition texte']
T7 Cas similaires en condition image détectés : ✓ PASS ['fuite:cas_similaires_en_condition_image']
T8 Prédiction manquante (null) acceptée : ✓ PASS []
T9 JSON LLM invalide → None : ✓ PASS extrait=None
T10 Appareil non standard non signalé détecté : ✓ PASS ['appareil_non_standard_non_signale']

RÉSULTAT : 10/10 tests passés
  ✓ T1_inversion_signalee
  ✓ T2_

## R7 : Génération des rapports médicaux structurés

In [11]:
# ============================================================
# R7 : Génération des rapports médicaux structurés
# Mode extractif pur (+ LLM si disponible)
# ============================================================
import collections

def get_vision_facts(d):
    """Extrait les faits vision pour un patient."""
    v = d.get("vision")
    if not v:
        return dict(condition=None, mammography_probabilities=None,
                    ultrasound_malignant_probability=None,
                    uncertainty_entropy=None, standard_device=True)
    return dict(
        mammography_probabilities=dict(
            normal=v["p_normal_mg"], benign=v["p_benign_mg"], malignant=v["p_malignant_mg"]
        ) if v["p_malignant_mg"] is not None else None,
        ultrasound_malignant_probability=v["p_malignant_us"],
        uncertainty_entropy=v["entropy_mg"],
        standard_device=v["standard_device"]
    )

def get_visual_explanations(pid, condition, d):
    """Retourne les explications XAI réelles ou 'indisponible' — jamais fabriquées."""
    if condition == "texte":
        return None
    hm = d.get("heatmap")
    if not hm:
        return {"mammography": "Grad-CAM heatmap: not available for this patient",
                "ultrasound":  "Ultrasound Grad-CAM: not available for this patient"}
    return build_heatmap_summary(hm)

def vision_interpretation_sentence(F, cond):
    """Phrase d'interprétation extractive basée strictement sur les données."""
    if cond == "texte" or not F.get("mammography_probabilities"):
        return "Text-only evaluation: no deep learning vision scores available."
    p = F["mammography_probabilities"]
    s = (f"Mammography model: normal={p.get('normal',0):.2f}, "
         f"benign={p.get('benign',0):.2f}, malignant={p.get('malignant',0):.2f} "
         f"(entropy={F.get('uncertainty_entropy',0):.2f}).")
    pu = F.get("ultrasound_malignant_probability")
    if pu is not None:
        s += f" Ultrasound malignant={pu:.2f}."
        pm = p.get("malignant", 0)
        if (pm >= 0.5) != (pu >= 0.5):
            s += " The two modality scores point in different directions."
        else:
            s += " Both modalities agree."
    if not F.get("standard_device", True):
        s += " Non-standard acquisition device: scores are less reliable."
    return s

def select_recommendations(pb, cond, pred_available):
    """Recommandations basées sur BI-RADS — sans règle de secours inventée."""
    if cond == "texte":
        return ["Clinical management per radiologist review of findings.",
                "Correlate with dedicated breast imaging."]
    if not pred_available or pb is None:
        return ["No vision prediction available. Human radiologist review required for BI-RADS assignment."]
    recs = {
        "1": ["Negative findings. Standard age-appropriate routine screening."],
        "2": ["Routine annual screening mammography."],
        "3": ["Short-interval 6-month follow-up imaging (unilateral)."],
        "4A":["Image-guided core needle biopsy (CNB) recommended — ACR 4A.",
              "Multidisciplinary breast tumor board review."],
        "4B":["Image-guided core needle biopsy (CNB) recommended — ACR 4B.",
              "Multidisciplinary breast tumor board review."],
        "4C":["Immediate image-guided core needle biopsy — ACR 4C.",
              "Surgical oncology consultation."],
        "5": ["Immediate core needle biopsy — ACR 5.",
              "Surgical oncology consultation."],
    }
    return recs.get(str(pb),
                    ["Additional diagnostic evaluation and clinical correlation required."])

def run_row(row, d):
    """Génère un rapport médical structuré pour une ligne de prompt."""
    F = get_vision_facts(d)

    # Prédiction BI-RADS et confiance
    if row.condition == "texte":
        pb, cf = None, None
        pred_available = False
    else:
        v = d.get("vision")
        pb = v["predicted_birads"] if v else None
        cf = v["confidence"]       if v else None
        pred_available = pb is not None

    # XAI
    vis_xai = get_visual_explanations(row.pseudo_id, row.condition, d)

    # Interprétation extractive
    interp = vision_interpretation_sentence(F, row.condition)

    # Recommandations
    recs = select_recommendations(pb, row.condition, pred_available)

    # Limitations
    lims = ["Research baseline AI models — not clinically validated as autonomous diagnostic tools.",
            "Human radiologist review required before any clinical decision."]
    if vis_xai and "not available" not in str(vis_xai):
        lims.append("Grad-CAM highlights network attention areas, not definitive tissue boundaries.")
    if not F.get("standard_device", True):
        lims.append("Non-standard acquisition hardware: image quality may be reduced.")
    if F.get("ultrasound_malignant_probability") is not None:
        lims.append("Ultrasound model is secondary; mammography carries primary weight.")

    # Guidelines et cas similaires
    gl_key = ("guidelines_image"     if row.condition == "image"  else
              "guidelines_texte"     if row.condition == "texte"  else
              "guidelines_multimodal")
    guidelines = [dict(citation=f"G{k}", rule_id=g["id"], source=g["source"],
                       rule=g["passage"][:160])
                  for k, g in enumerate(d.get(gl_key, []), 1)]
    cases = [dict(citation=f"E{k}", source=e["source_patient"],
                  similarity=e["score"], passage=e["passage"][:200])
             for k, e in enumerate(d.get("evidence", []), 1)
             if row.condition != "image"]

    # Rapport final
    report = dict(
        patient_summary=dict(
            pseudo_id        = row.pseudo_id,
            condition        = row.condition,
            acquisition_device = ("Standard" if F.get("standard_device", True)
                                 else "Non-standard (reduced resolution)")
        ),
        predicted_birads    = pb,
        prediction_available= pred_available,
        confidence          = cf,
        confidence_note     = ("Raw model value — not clinically calibrated"
                               if cf is not None else None),
        interpretation      = interp,
        supporting_evidence = [g["citation"] + f" [{g['rule_id']}] {g['rule'][:120]}"
                               for g in guidelines],
        similar_cases       = cases,
        visual_explanations = vis_xai,
        recommendations     = recs,
        limitations         = lims,
        reasoning_mode      = "extractive" if not LLM_ENABLED else "llm_attempted",
        human_review_required = True
    )

    # Optionnel : LLM
    llm_output = None
    if LLM_ENABLED:
        try:
            raw = generate_llm(row.system, row.user)
            llm_output = extract_json(raw)
            if llm_output:
                report["llm_output"] = llm_output
                report["reasoning_mode"] = "llm_succeeded"
            else:
                report["llm_raw"] = raw[:500] if raw else None
                report["reasoning_mode"] = "llm_failed_extractive_fallback"
        except Exception as e:
            report["llm_error"] = str(e)
            report["reasoning_mode"] = "llm_error_extractive_fallback"

    # Contrôle qualité exhaustif
    F["standard_device"] = bool(d.get("vision", {}).get("standard_device", True) if d.get("vision") else True)
    rep_text = d.get("findings", "") or ""
    errs = validate_medical_report(report, row, F, report_text=rep_text)

    # Statuts explicites (jamais auto-déclarés sans preuve)
    n_structural = sum(1 for e in errs if "champ_manquant" in e or "invalide" in e)
    n_isolation  = sum(1 for e in errs if "fuite" in e)
    n_content    = len(errs) - n_structural - n_isolation

    if not errs:
        safety_status = "checks_passed"
    elif n_isolation > 0:
        safety_status = "isolation_violation_review_required"
    else:
        safety_status = "review_required"

    report["quality_safety_check"] = dict(
        validation_errors      = errs,
        safety_status          = safety_status,
        # Ces champs sont "not_evaluated" sauf preuve directe
        groundedness           = "not_evaluated",
        hallucination_detection= "not_evaluated",
        birads_consistency     = ("inconsistent" if any("birads" in e.lower() or
                                                        "discordance" in e for e in errs)
                                  else "checks_passed"),
        modality_isolation     = ("violated" if n_isolation > 0 else "checks_passed"),
        human_review_required  = True,
        human_review_note      = "Automated checks only — radiologist review mandatory for any clinical use"
    )

    return report

# ── Génération sur un échantillon représentatif ──
print("Génération des rapports médicaux structurés...")

def safe_sample(df, n, seed=0):
    """sample() sûr même sur une DataFrame vide ou trop petite."""
    if len(df) == 0: return df
    return df.sample(min(n, len(df)), random_state=seed)

pm_les_deux = safe_sample(Pm[(Pm.condition=="les_deux") & Pm.has_report], 12)
pm_texte    = safe_sample(Pm[(Pm.condition=="texte")    & Pm.has_report], 4)
pm_image    = safe_sample(Pm[ Pm.condition=="image"    ], 4)

sample_rows = (list(pm_les_deux.itertuples()) +
               list(pm_texte.itertuples()) +
               list(pm_image.itertuples()))

print(f"Échantillon : {len(pm_les_deux)} multimodal + {len(pm_texte)} texte + {len(pm_image)} image")

final_reports = []
for row in sample_rows:
    try:
        d = D.get(row.pseudo_id)
        if d is None:
            print(f"  [AVERTISSEMENT] {row.pseudo_id} : dossier absent")
            continue
        final_reports.append(run_row(row, d))
    except Exception as e:
        print(f"  [ERREUR] {row.pseudo_id}/{row.condition} : {e}")

print(f"\nRapports générés : {len(final_reports)}")
conds = collections.Counter(r["patient_summary"]["condition"] for r in final_reports)
print(f"Distribution : {dict(conds)}")

# Bilans de conformité
n_passed   = sum(1 for r in final_reports if r["quality_safety_check"]["safety_status"] == "checks_passed")
n_txt_iso  = sum(1 for r in final_reports
                 if r["patient_summary"]["condition"] == "texte"
                 and r["predicted_birads"] is None
                 and r["visual_explanations"] is None)
n_img_iso  = sum(1 for r in final_reports
                 if r["patient_summary"]["condition"] == "image"
                 and not r.get("similar_cases"))

print(f"\n── Bilan de conformité ──")
print(f"  Contrôles passés    : {n_passed}/{len(final_reports)}")
print(f"  Texte étanche (0 vision, 0 XAI) : {n_txt_iso}/{conds.get('texte',0)}")
print(f"  Image étanche (0 cas similaires) : {n_img_iso}/{conds.get('image',0)}")

# Exemples
for label, cond in [("MULTIMODAL", "les_deux"), ("TEXTE SEUL", "texte"), ("IMAGE SEULE", "image")]:
    ex = next((r for r in final_reports if r["patient_summary"]["condition"] == cond), None)
    if ex:
        print(f"\n── Exemple : {label} ──")
        print(json.dumps({k: ex[k] for k in
                          ["patient_summary","predicted_birads","interpretation",
                           "recommendations","quality_safety_check"]},
                         ensure_ascii=False, indent=2))

Génération des rapports médicaux structurés...
Échantillon : 12 multimodal + 4 texte + 4 image

Rapports générés : 20
Distribution : {'les_deux': 12, 'texte': 4, 'image': 4}

── Bilan de conformité ──
  Contrôles passés    : 17/20
  Texte étanche (0 vision, 0 XAI) : 4/4
  Image étanche (0 cas similaires) : 4/4

── Exemple : MULTIMODAL ──
{
  "patient_summary": {
    "pseudo_id": "P0193",
    "condition": "les_deux",
    "acquisition_device": "Standard"
  },
  "predicted_birads": "3",
  "interpretation": "Mammography model: normal=0.31, benign=0.62, malignant=0.07 (entropy=0.77). Ultrasound malignant=0.52. The two modality scores point in different directions.",
  "recommendations": [
    "Short-interval 6-month follow-up imaging (unilateral)."
  ],
  "quality_safety_check": {
    "validation_errors": [],
    "safety_status": "checks_passed",
    "groundedness": "not_evaluated",
    "hallucination_detection": "not_evaluated",
    "birads_consistency": "checks_passed",
    "modality_isol

## R8 : Export final — JSON + CSV + Résumé des corrections

In [12]:
# ============================================================
# R8 : Export final — JSON + CSV + Résumé des corrections
# ============================================================

# ── Exports principaux ──
json.dump(final_reports, open(str(WORK_DIR / "final_medical_reports.json"), "w",
                              encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"✓ final_medical_reports.json ({len(final_reports)} rapports)")

# Export CSV récapitulatif
csv_rows = []
for r in final_reports:
    qs = r["quality_safety_check"]
    csv_rows.append(dict(
        pseudo_id          = r["patient_summary"]["pseudo_id"],
        condition          = r["patient_summary"]["condition"],
        predicted_birads   = r.get("predicted_birads"),
        prediction_available = r.get("prediction_available"),
        confidence         = r.get("confidence"),
        safety_status      = qs["safety_status"],
        n_errors           = len(qs["validation_errors"]),
        modality_isolation = qs["modality_isolation"],
        reasoning_mode     = r.get("reasoning_mode"),
        n_recommendations  = len(r.get("recommendations",[])),
        n_guidelines       = len(r.get("supporting_evidence",[])),
        n_similar_cases    = len(r.get("similar_cases",[])),
        errors_list        = " | ".join(qs["validation_errors"])
    ))
pd.DataFrame(csv_rows).to_csv(str(WORK_DIR / "final_reports_summary.csv"), index=False)
print("✓ final_reports_summary.csv")

# Export des résultats des 10 tests adversariaux
json.dump(RESULTS, open(str(WORK_DIR / "adversarial_test_results.json"), "w",
                        encoding="utf-8"), ensure_ascii=False, indent=2)
print("✓ adversarial_test_results.json")

# ── Résumé des corrections apportées ──
CORRECTIONS_SUMMARY = {
    "version": "v3",
    "date": "2026-10-15",
    "corrections": {
        "1_birads_confiance": [
            "Suppression de la note 'calibrée' sur la confiance — remplacée par 'non calibrée cliniquement'",
            "predicted_birads=None si valeur brute invalide — motif explicite fourni",
            "Aucune catégorie fabriquée par règle de secours"
        ],
        "2_isolation_modalites": [
            "Condition image : zéro texte, zéro cas similaires — vérifié mathématiquement",
            "Condition texte : zéro vision, zéro heatmap — vérifié mathématiquement",
            "3 requêtes RAG séparées : query_img, query_txt, query_both",
        ],
        "3_rag_exclusion": [
            "search_cases() exclut : même patient + même groupe + même pli",
            "group_pseudo ajouté aux chunks C pour permettre l'exclusion de groupe",
            "Test automatique : 0 violation sur 30 patients × 5 résultats"
        ],
        "4_raisonnement": [
            "vision_interpretation_sentence() : probabilités brutes inchangées",
            "Mode extractif explicitement signalé (reasoning_mode='extractive')",
            "LLM optionnel avec fallback extractif explicite si échec",
            "Sortie LLM invalide → extract_json() retourne None → fallback"
        ],
        "5_controle_qualite": [
            "validate_medical_report() : validation de TOUS les champs SCHEMA",
            "Statuts explicites : checks_passed | review_required | not_evaluated",
            "groundedness/hallucinations = 'not_evaluated' (jamais auto-déclarés)",
            "Inversion mammo/écho vérifiée (heuristique)",
            "Discordance obligatoirement signalée si pm≥0.5 XOR pu≥0.5"
        ],
        "6_xai": [
            "build_heatmap_summary() : retourne null si heatmap absente",
            "Aucune phrase générique de secours inventée",
            "zone_desc() : calcul à partir des coordonnées réelles",
            "Détection des phrases XAI génériques fabriquées"
        ],
        "7_kaggle": [
            "Cellule R0 de diagnostic préalable des fichiers et dépendances",
            "safe_sample() : ne crashe pas sur une catégorie vide",
            "Chemins centralisés dans WORK_DIR + SEARCH_BASES",
            "CPU/GPU auto-détecté, LLM facultatif"
        ]
    },
    "limites_restantes": [
        "Détection d'inversion mammo/écho heuristique (non exhaustive)",
        "Guidelines ACR hardcodées (résumés éditoriaux, non PDFs officiels)",
        "LLM non testé sur Kaggle dans cette session",
        "Calibration de la confiance non vérifiable sans données de calibration",
        "Cohérence BI-RADS / probabilités non validée cliniquement"
    ],
    "tests_adversariaux": {k: v["status"] for k, v in RESULTS.items()},
    "tests_passes": sum(1 for v in RESULTS.values() if v["status"] == "PASS"),
    "tests_total": len(RESULTS)
}
json.dump(CORRECTIONS_SUMMARY, open(str(WORK_DIR / "corrections_summary.json"), "w",
                                     encoding="utf-8"), ensure_ascii=False, indent=2)
print("✓ corrections_summary.json")

# ── Résumé final ──
print("\n" + "="*65)
print("EXPORTS FINAUX — /kaggle/working/")
for f in sorted(WORK_DIR.glob("*.json")) + sorted(WORK_DIR.glob("*.csv")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size:6.1f} Ko")
print("="*65)
print(f"\nTests adversariaux : {CORRECTIONS_SUMMARY['tests_passes']}/{CORRECTIONS_SUMMARY['tests_total']} PASS")
print(f"Rapports générés   : {len(final_reports)}")
print(f"Contrôles passés   : {n_passed}/{len(final_reports)}")
print("\n[AVERTISSEMENT] Ce notebook est un prototype de recherche.")
print("Toutes les sorties requièrent la revue d'un radiologue qualifié.")

✓ final_medical_reports.json (20 rapports)
✓ final_reports_summary.csv
✓ adversarial_test_results.json
✓ corrections_summary.json

EXPORTS FINAUX — /kaggle/working/
  adversarial_test_results.json                    1.2 Ko
  corrections_summary.json                         2.9 Ko
  final_medical_reports.json                      80.3 Ko
  acr_guidelines_chunks.csv                        3.4 Ko
  final_reports_summary.csv                        1.8 Ko
  rag_chunks.csv                                 613.2 Ko

Tests adversariaux : 10/10 PASS
Rapports générés   : 20
Contrôles passés   : 17/20

[AVERTISSEMENT] Ce notebook est un prototype de recherche.
Toutes les sorties requièrent la revue d'un radiologue qualifié.


## Instructions Kaggle

### Inputs requis
| Dataset | Fichiers | Rôle |
|---------|----------|------|
| `data-quality-clean` | `reports_anon/*.json`, `dataset_final.csv` | Rapports patients nettoyés |
| `agent-vision22` | `predictions_patient.csv`, `heatmap_fiches_pli0.json` | Prédictions vision + heatmaps |

### Configuration
- Accélérateur : **GPU T4 x1** (ou CPU — LLM désactivé automatiquement)
- Internet : **Activé** (téléchargement `bge-small-en` + `Qwen2.5-3B` au premier run)
- Session : **≥ 3h** avec GPU, **≥ 6h** sans

### Exécution
1. Ajouter les deux datasets comme inputs
2. Cliquer **Run All**
3. Vérifier les sorties dans `/kaggle/working/`

### Sorties produites
- `final_medical_reports.json` — rapports structurés
- `final_reports_summary.csv` — tableau récapitulatif
- `adversarial_test_results.json` — résultats des 10 tests
- `corrections_summary.json` — bilan des corrections et limites
- `prompts.jsonl`, `dossiers_patients.jsonl` — données intermédiaires
